In [ ]:
# Pandas  

# Pandas

🔹 What it is
* A Python library for data manipulation and analysis
* Works with DataFrames (like Excel tables)

🔹 Key Features

* Easy to use and beginner-friendly
* Fast for small to medium datasets
* Rich functions for cleaning, filtering, grouping, etc.

🔹 Limitations

* Runs on a single machine (RAM-bound)
* Not suitable for big data (GBs–TBs)

🔹 Use Cases

* Data cleaning
* Exploratory Data Analysis (EDA)
* Machine learning preprocessing


## 2. MapReduce
* What it is
    
* A programming model for processing large datasets in parallel
Popularized by Apache Hadoop

🔹 How it Works

* Map → Process and transform data

* Shuffle → Group intermediate results

* Reduce → Aggregate final output

🔹 Key Features

* Handles very large data (TBs–PBs)

* Fault-tolerant (handles failures automatically)

🔹 Limitations

* Slow (disk-based processing)

* Complex to write and debug

* Not interactive

🔹 Use Cases

* Batch processing

* Log analysis

* Large-scale aggregation

# 3. Apache Spark

## PySpark Notebook: Smart Mobility + Parallelization Deep Dive

### Start Spark Session

In [21]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Smart Mobility Analysis") \
    .getOrCreate()

spark

### SparkSession is the entry point.

* Internally:
    * - Driver program starts
    * - Connects to cluster (or local mode)
    * - Manages execution of jobs

#### Load CSV using Spark

In [22]:
df = spark.read.csv(
    "smart_mobility_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|          Timestamp|          Latitude|         Longitude|Vehicle_Count| Traffic_Speed_kmh| Road_Occupancy_%|Traffic_Light_State|Weather_Condition|Accident_Report|     Sentiment_Score|Ride_Sharing_Demand|Parking_Availability|Emission_Levels_g_km|Energy_Consumption_L_h|Traffic_Condition|
+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|2024-03-01 00:00:00|40.842275292891834|-73.70314869323049|          205| 49.89343479610332|82.65277992850861|             Yellow|   

### What happens internally:

     1. File is split into partitions
     2. Each partition is sent to a worker node
     3. Each worker reads its chunk in parallel

* This is where parallelism starts

### Check Schema

In [3]:
df.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Vehicle_Count: integer (nullable = true)
 |-- Traffic_Speed_kmh: double (nullable = true)
 |-- Road_Occupancy_%: double (nullable = true)
 |-- Traffic_Light_State: string (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Accident_Report: integer (nullable = true)
 |-- Sentiment_Score: double (nullable = true)
 |-- Ride_Sharing_Demand: integer (nullable = true)
 |-- Parking_Availability: integer (nullable = true)
 |-- Emission_Levels_g_km: double (nullable = true)
 |-- Energy_Consumption_L_h: double (nullable = true)
 |-- Traffic_Condition: string (nullable = true)



#### Spark automatically infers:
    * - Integer
    * - Double
    * - String
    * - Timestamp (if formatted)

* Schema helps Catalyst Optimizer optimize execution

### Check Partitions

In [4]:
df.rdd.getNumPartitions()

1

* Partition = unit of parallelism

* Example:
    * - 4 partitions → 4 tasks run in parallel

* More partitions = more parallel execution (up to cluster limit)

#### Repartition (Control Parallelism)

In [5]:
df = df.repartition(4)

df.rdd.getNumPartitions()

[Stage 3:===========================================================(1 + 0) / 1]

4

### Repartition:
    * - Redistributes data across nodes
    * - Increases parallelism
    * - Causes shuffle (expensive)

* Use carefully!

### Transformations (Lazy Execution)

In [6]:
df_filtered = df.filter(df.Vehicle_Count > 200)

### IMPORTANT 🔥

* This does NOT execute immediately.

* Spark builds a DAG (Directed Acyclic Graph):
    * - Records transformation steps
    * - Waits for an action

* This is called LAZY EVALUATION

#### Trigger Action

In [24]:
df_filtered.show(3)

[Stage 51:>                                                         (0 + 1) / 1]

+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|          Timestamp|          Latitude|         Longitude|Vehicle_Count| Traffic_Speed_kmh| Road_Occupancy_%|Traffic_Light_State|Weather_Condition|Accident_Report|     Sentiment_Score|Ride_Sharing_Demand|Parking_Availability|Emission_Levels_g_km|Energy_Consumption_L_h|Traffic_Condition|
+-------------------+------------------+------------------+-------------+------------------+-----------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|2024-03-13 03:50:00| 40.79043297437304|-73.73217633460834|          256|47.660479949669224|29.54162316852753|              Green|   

### Now execution starts:

* 1. DAG is optimized (Catalyst)
* 2. Job is created
* 3. Split into stages
* 4. Tasks sent to workers

#### Map Transformation (RDD Level)

In [8]:
rdd = df.rdd

mapped = rdd.map(lambda row: (row["Vehicle_Count"], row["Traffic_Speed_kmh"]))

mapped.take(5)

[(189, 57.95185679606976),
 (198, 22.990604151182517),
 (216, 6.604043772852476),
 (59, 18.926178982770303),
 (60, 25.019430607584702)]

#### map() runs in parallel:

* Each partition:
 - Applies lambda function independently

### Example:
* Partition 1 → rows 1–100  
* Partition 2 → rows 101–200  

* Each runs on different CPU cores

### flatMap Example

In [9]:
rdd_text = spark.sparkContext.parallelize(["heavy traffic now", "smooth road"])

words = rdd_text.flatMap(lambda x: x.split(" "))
words.collect()

['heavy', 'traffic', 'now', 'smooth', 'road']

### flatMap:
     - Splits each record into multiple outputs
     - Still runs in parallel across partitions

### Filter Transformation

In [10]:
high_traffic = df.filter(df.Traffic_Condition == "High")
high_traffic.count()

3166

* Each partition filters its own data independently

* No communication needed → very fast

#### Shuffle Operation (Important ⚠️)

In [11]:
df.groupBy("Traffic_Condition").count().show()

[Stage 18:>                                                         (0 + 4) / 4]

+-----------------+-----+
|Traffic_Condition|count|
+-----------------+-----+
|             High| 3166|
|              Low|  359|
|           Medium| 1475|
+-----------------+-----+



#### This triggers a SHUFFLE:

 - Data is redistributed across nodes
 - All "High" go together, "Low" together

#### Shuffle is expensive because:
 - Network transfer
 - Disk I/O

* Avoid excessive shuffles!

#### Convert Timestamp

In [12]:
from pyspark.sql.functions import col, hour

df = df.withColumn("Hour", hour(col("Timestamp")))
df.show(5)

[Stage 22:>                                                         (0 + 1) / 1]

+-------------------+------------------+------------------+-------------+------------------+------------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+----+
|          Timestamp|          Latitude|         Longitude|Vehicle_Count| Traffic_Speed_kmh|  Road_Occupancy_%|Traffic_Light_State|Weather_Condition|Accident_Report|     Sentiment_Score|Ride_Sharing_Demand|Parking_Availability|Emission_Levels_g_km|Energy_Consumption_L_h|Traffic_Condition|Hour|
+-------------------+------------------+------------------+-------------+------------------+------------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+----+
|2024-03-04 17:15:00| 40.60765736905612|-73.84292976843938|          189| 57.95185679606976| 74.84442993972618|    

### Column transformations are also parallel:

#### Each partition:
 - Applies transformation independently

#### Aggregation (Parallel Reduce)

In [13]:
df.groupBy("Hour").avg("Vehicle_Count").show()

[Stage 27:>                                                         (0 + 4) / 4]

+----+------------------+
|Hour|avg(Vehicle_Count)|
+----+------------------+
|  12|152.64705882352942|
|  22|151.37254901960785|
|   1|160.69444444444446|
|  13| 153.5735294117647|
|   6|153.39814814814815|
|  16|             154.5|
|   3|159.44907407407408|
|  20| 152.0735294117647|
|   5|158.27314814814815|
|  19|171.13235294117646|
|  15|145.02450980392157|
|  17| 156.9607843137255|
|   9|142.85294117647058|
|   4|152.65277777777777|
|   8|156.48584905660377|
|  23|146.97549019607843|
|   7|153.77314814814815|
|  10|162.86274509803923|
|  21|151.80392156862746|
|  11| 151.6764705882353|
+----+------------------+
only showing top 20 rows



#### How this works internally:

1. Each partition computes partial average
2. Results are combined (reduce phase)

➡️ MapReduce style execution

#### Caching (Performance Boost)

In [15]:
print("df_cache",df.cache())
print("df_count",df.count())

26/03/27 09:31:27 WARN CacheManager: Asked to cache already cached data.


df_cache DataFrame[Timestamp: timestamp, Latitude: double, Longitude: double, Vehicle_Count: int, Traffic_Speed_kmh: double, Road_Occupancy_%: double, Traffic_Light_State: string, Weather_Condition: string, Accident_Report: int, Sentiment_Score: double, Ride_Sharing_Demand: int, Parking_Availability: int, Emission_Levels_g_km: double, Energy_Consumption_L_h: double, Traffic_Condition: string, Hour: int]
df_count 5000


#### Caching:
- Stores data in memory
- Avoids recomputation

Useful when:
- Reusing same DataFrame multiple times

## Explain Execution Plan

In [17]:
## df.groupBy("Traffic_Condition").count().explain()

### Shows:
- Logical plan
- Physical plan
- Optimizations applied

This is Spark’s brain (Catalyst Optimizer)

## Spark SQL

### Spark SQL uses same engine:
- Same optimizations
- Same parallel execution

#### Parallelization Demo (Core Concept)

In [23]:
numbers = spark.sparkContext.parallelize(range(1, 10000000), 4)

result = numbers.map(lambda x: x * 2).sum()

result

99999990000000

What happens:

1. Data split into 4 partitions
2. Each partition processed on different core
3. map() runs in parallel
4. sum() aggregates results

➡️ This is TRUE parallel computing

#### Global vs Local Scope

In [19]:
x = 10

rdd = spark.sparkContext.parallelize([1,2,3])

rdd.map(lambda y: y + x).collect()

[11, 12, 13]

Spark sends variables (x) to workers.

But:
- Large variables → use broadcast variables

#### Broadcast Variable

In [20]:
broadcast_var = spark.sparkContext.broadcast(10)

rdd.map(lambda y: y + broadcast_var.value).collect()

[11, 12, 13]

Broadcast:
- Sends data once to all workers
- Efficient for large shared variables

### Summary
🔥 Key Concepts Learned:

✔ Spark reads data in parallel using partitions  
✔ Transformations are lazy (build DAG)  
✔ Actions trigger execution  
✔ Each partition runs independently  
✔ Shuffle = expensive operation  
✔ Aggregations use MapReduce pattern  
✔ Parallelization = multiple tasks across cores  

💡 Rule:
More partitions → more parallelism (but more overhead)

This is how Spark scales to BIG DATA.